In [ ]:
import os
import torch
import einops
import random
import argparse
import math

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

from scipy.special import erf
from scipy.fft import rfft, rfftfreq
from dataclasses import replace
from omegaconf import OmegaConf, ListConfig
from pathlib import Path
from torch.utils.data import DataLoader

from utils.config import *
from utils.einmask import EinMask
from utils.dataset import NinoData, MultifileNinoDataset


class Evaluation:
    def __init__(self, cfg: MTMConfig, task: str = 'frcst', save_mode: str = 'none'):
        self._cfg = cfg
        self.task = task
        self.save_mode = save_mode
        self.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

        self.create_paths()
        self.generator = self.create_seed()
        self.val_dataset, self.val_dl = self.create_dataset()
        self.model = self.create_model()

    # SETUP
    def create_paths(self):
        self.model_dir = Path(self.cfg.model_dir) / self.cfg.job_name
        self.ckpt_path = self.model_dir / 'ckpt.pth'
        self.best_path = self.model_dir / 'best.pth'
        self.cfg_path  = self.model_dir / 'config.yaml'

    def create_model(self):
        model = EinMask(self.model_cfg, self.world)
        ckpt  = torch.load(self.best_path, map_location='cpu')
        state = {k.removeprefix('module.'): v for k, v in ckpt['model_state'].items()}
        model.load_state_dict(state)
        return model.to(self.device)

    def create_dataset(self):
        val_dataset = getattr(self, f'{self.data_cfg.eval_data.lower()}_data')()
        self._lsm = einops.repeat(
            torch.logical_not(val_dataset.land_sea_mask.to(dtype=torch.bool)),
            f'1 (h hh) (w ww) -> {self.world.field_pattern}',
            **self.world.token_sizes, **self.world.patch_sizes
        ).to(self.device)
        val_dl = DataLoader(val_dataset, shuffle=False,
                            batch_size=self.world.batch_size, drop_last=True)
        return val_dataset, val_dl

    def create_seed(self):
        if not exists(self.cfg.seed):
            seed = int.from_bytes(random.randbytes(4), byteorder='little')
            self.cfg.seed = seed
        np.random.seed(self.cfg.seed)
        random.seed(self.cfg.seed)
        torch.manual_seed(self.cfg.seed)
        torch.cuda.manual_seed(self.cfg.seed)
        return torch.Generator(device=self.device).manual_seed(self.cfg.seed)

    # DATA
    def lens_data(self):
        if not hasattr(self, '_lens_data'):
            lens_config = replace(self.data_cfg,
                time_slice={'start': '1850', 'stop': '2000', 'step': None},
                stats=default(self.data_cfg.stats, LENS_STATS))
            self._lens_data = MultifileNinoDataset(self.cfg.lens_path, lens_config, 0, 1)
        return self._lens_data

    def godas_data(self):
        if not hasattr(self, '_godas_data'):
            godas_config = replace(self.data_cfg,
                time_slice={'start': '1980', 'stop': '2020', 'step': None},
                stats=default(self.data_cfg.stats, GODAS_STATS))
            self._godas_data = NinoData(self.cfg.godas_path, godas_config)
        return self._godas_data

    def picontrol_data(self):
        if not hasattr(self, '_picontrol_data'):
            picontrol_config = replace(self.data_cfg,
                time_slice={'start': '1900', 'stop': '2000', 'step': None},
                stats=default(self.data_cfg.stats, PICONTROL_STATS))
            self._picontrol_data = NinoData(self.cfg.picontrol_path, picontrol_config)
        return self._picontrol_data

    def oras5_data(self):
        if not hasattr(self, '_oras5_data'):
            oras5_config = replace(self.data_cfg,
                time_slice={'start': '1980', 'stop': '2020', 'step': None})
            self._oras5_data = NinoData(self.cfg.oras5_path, oras5_config)
        return self._oras5_data

    # CONFIG
    @property
    def cfg(self) -> TrainerConfig:
        return self._cfg.trainer

    @property
    def data_cfg(self) -> DatasetConfig:
        return self._cfg.data

    @property
    def model_cfg(self) -> NetworkConfig:
        return self._cfg.model

    @property
    def world(self) -> WorldConfig:
        return self._cfg.world

    @property
    def objective(self) -> ObjectiveConfig:
        return self._cfg.objective

    # MASKING
    def sample_normal_rates_(self, mean: float, std: float, a: float = 0., b: float = 1.):
        return torch.nn.init.trunc_normal_(
            torch.empty((1,), device=self.device),
            mean=mean, std=std, a=a, b=b, generator=self.generator
        ).mul(self.world.num_tokens).long()

    def sample_weighted_reservoir(self, num_samples: int):
        P = torch.rand((num_samples, self.world.num_tokens),
                       device=self.device, generator=self.generator).log()
        for dim, alpha in self.objective.event_cfg.items():
            if not (exists(alpha) and dim in self.world.layout): continue
            U = torch.rand((num_samples, self.world.token_sizes[dim]),
                           device=self.device, generator=self.generator)
            U = einops.repeat(U, f'b {dim} -> b ({self.world.token_pattern})',
                              **self.world.token_sizes)
            P += U.log().div(alpha)
        return P

    def sample_block_noise(self, K: int, num_samples: int):
        d = torch.arange(1, K + 1, device=self.device)
        d = d[K % d == 0]
        idx = torch.multinomial(1 / d, 1, generator=self.generator)
        KK = d[idx]
        U = torch.rand((num_samples, K // KK), device=self.device, generator=self.generator)
        U = einops.repeat(U, f'... k -> ... (k kk)', kk=KK, k=K // KK)
        return U

    def sample_weighted_reservoir_blocks(self, num_samples: int):
        P = torch.rand((num_samples, self.world.num_tokens), device=self.device, generator=self.generator).log()
        for dim, alpha in self.objective.event_cfg.items():
            if not (exists(alpha) and dim in self.world.layout): continue
            U = self.sample_block_noise(self.world.token_sizes[dim], num_samples)
            U = einops.repeat(U, f'b {dim} -> b ({self.world.token_pattern})', **self.world.token_sizes)
            P += U.log().div(alpha)
        return P

    def sample_masks(self) -> torch.BoolTensor:
        K_src = self.sample_normal_rates_(**self.objective.rate_cfg)
        src_weights = self.sample_weighted_reservoir(self.world.batch_size)
        src_reservoir = src_weights.argsort(descending=True)
        return src_reservoir.argsort(descending=False).lt(K_src)

    @property
    def frcst_prefix(self) -> torch.BoolTensor:
        prefix = torch.zeros((self.world.token_sizes['t'],), device=self.device, dtype=torch.bool)
        prefix[:self.world.tau] = True
        return einops.repeat(prefix, f't -> b ({self.world.token_pattern})',
                             **self.world.token_sizes, b=self.world.batch_size)

    @property
    def intervention_prefix(self) -> torch.BoolTensor:
        indices = [self.data_cfg.variables.index(var) for var in self.data_cfg.eval_variables]
        prefix  = torch.zeros((self.world.token_sizes['v'],), device=self.device, dtype=torch.bool)
        prefix[indices] = True
        return einops.repeat(prefix, f'v -> b ({self.world.token_pattern})',
                             **self.world.token_sizes, b=self.world.batch_size)

    def make_visible(self) -> torch.BoolTensor:
        if self.task.startswith('frcst_intervention'):
            return self.frcst_prefix & self.intervention_prefix
        elif self.task.startswith('frcst_zeros'):
            return self.frcst_prefix
        elif self.task.startswith('frcst'):
            return self.frcst_prefix
        elif self.task == 'masked':
            return self.sample_masks()
        else:
            raise NotImplementedError(f'unknown task: {self.task}')

    # FORWARD
    def to_xarray_eval(self, batch_idx, obs, mu, sigma, visible) -> xr.Dataset:
        meta = self.val_dataset.dataset
        batch_slice = slice(batch_idx * obs.shape[0], (batch_idx + 1) * obs.shape[0])
        vis_field = einops.repeat(
            visible,
            f'... ({self.world.token_pattern}) -> ... {self.world.field_pattern}',
            **self.world.token_sizes, **self.world.patch_sizes
        )
        arrays = []
        for v, var in enumerate(self.data_cfg.variables):
            if var not in self.data_cfg.eval_variables:
                continue
            std = self.val_dataset._stds.sel(variable=var).values.astype(np.float32)
            arrays.append(xr.Dataset({
                f'{var}_obs':(['time', 'step', 'lat', 'lon'], obs[:, v].numpy() * std),
                f'{var}_pred_mu':(['time', 'step', 'lat', 'lon'], mu[:, v].numpy() * std),
                f'{var}_pred_sigma': (['time', 'step', 'lat', 'lon'], sigma[:, v].numpy() * std),
                f'{var}_visible':(['time', 'step', 'lat', 'lon'], vis_field[:, v].numpy()),
            }, coords={
                'time': meta.time[batch_slice],
                'step': np.arange(self.data_cfg.sequence_length),
                'lat':  meta.lat,
                'lon':  meta.lon,
            }))
        return xr.merge(arrays, compat='no_conflicts')

    def compute_metrics(self, ds: xr.Dataset) -> xr.Dataset:
        arrays = []
        for var in self.data_cfg.eval_variables:
            valid = ~ds[f'{var}_visible'] & ~ds['lsm']
            mu = ds[f'{var}_pred_mu'].where(valid)
            sigma = ds[f'{var}_pred_sigma'].where(valid)
            obs = ds[f'{var}_obs'].where(valid)

            arrays.append(xr.Dataset({
                f'pcc_{var}': self.xr_pcc(mu, obs, ['lat', 'lon']).mean('time'),
                f'rmse_{var}': self.xr_rmse(mu, obs, ['lat', 'lon']).mean('time'),
                f'rmse_ss_{var}': self.xr_rmse_ss(mu, obs, ['lat', 'lon']).mean('time'),
                f'ssr_{var}':self.xr_spread_skill_mve(mu, sigma, obs, ['lat', 'lon']).mean('time'),
                f'activ_{var}':self.xr_activ(mu, obs, ['lat', 'lon']).mean('time'),
                f'crps_{var}':self.xr_gaussian_crps(mu, sigma, obs).mean(['lat', 'lon', 'time']),
                f'crps_ss_{var}': self.xr_crps_ss(mu, sigma, obs, ['lat', 'lon']).mean('time'),
                f'ign_{var}':self.xr_gaussian_ign(mu, sigma, obs).mean(['lat', 'lon', 'time']),
                f'rmse_t_{var}':self.xr_rmse(mu, obs, ['lat', 'lon']),
                f'crps_t_{var}':self.xr_gaussian_crps(mu, sigma, obs).mean(['lat', 'lon']),
            }))

            if self.include_nino and var == 'temp_ocn_0a':
                for region, fn in [('nino34', self.get_nino34), ('nino4', self.get_nino4)]:
                    mu_r, sigma_r, obs_r = fn(mu), fn(sigma), fn(obs)
                    arrays.append(xr.Dataset({
                        f'pcc_{region}':self.xr_pcc(mu_r, obs_r, ['time']),
                        f'rmse_{region}':self.xr_rmse(mu_r, obs_r, ['time']),
                        f'rmse_ss_{region}': self.xr_rmse_ss(mu_r, obs_r, ['time']),
                        f'ssr_{region}':self.xr_spread_skill_mve(mu_r, sigma_r, obs_r, ['time']),
                        f'activ_{region}':self.xr_activ(mu_r, obs_r, ['time']),
                        f'crps_{region}':self.xr_gaussian_crps(mu_r, sigma_r, obs_r).mean('time'),
                        f'crps_ss_{region}': self.xr_crps_ss(mu_r, sigma_r, obs_r, ['time']),
                        f'ign_{region}':self.xr_gaussian_ign(mu_r, sigma_r, obs_r).mean('time'),
                    }))

        return xr.merge(arrays, compat='no_conflicts')

    def run(self) -> xr.Dataset:
        results = []
        self.model.eval()
        with torch.no_grad():
            for batch_idx, batch in enumerate(self.val_dl):
                batch = batch.to(self.device)
                visible = self.make_visible().to(self.device)

                if self.task.startswith('frcst_zeros'):
                    zeros = einops.repeat(
                        self.intervention_prefix,
                        f'... ({self.world.token_pattern}) -> ... {self.world.field_pattern}',
                        **self.world.token_sizes, **self.world.patch_sizes
                    )
                    batch = batch * zeros

                mu, sigma = self.model(batch, visible).mul(self._lsm)
                sigma = torch.nn.functional.softplus(sigma)
                results.append(self.to_xarray_eval(
                    batch_idx, batch.cpu(), mu.cpu(), sigma.cpu(), visible.cpu()
                ))

        raw_ds = xr.concat(results, dim='time', data_vars='all')
        raw_ds['lsm'] = xr.DataArray(
            np.bool_(self.val_dataset.land_sea_mask[0]),
            coords={'lat': self.val_dataset.dataset.lat, 'lon': self.val_dataset.dataset.lon}
        )
        metrics_ds = self.compute_metrics(raw_ds)
        self.write_to_disk(raw_ds, metrics_ds)
        return xr.merge([raw_ds, metrics_ds])

    def write_to_disk(self, raw_ds: xr.Dataset, metrics_ds: xr.Dataset):
        if self.save_mode == 'none':
            return
        base = self.model_dir / f'{self.data_cfg.eval_data}_{self.task}'
        if self.save_mode == 'metrics':
            metrics_ds.to_zarr(base.with_name(base.name + '_metrics.zarr'), mode='w')
        elif self.save_mode == 'all':
            xr.merge([raw_ds, metrics_ds]).to_zarr(base.with_name(base.name + '_eval.zarr'), mode='w')

    # METRICS
    @staticmethod
    def get_nino4(da: xr.DataArray) -> xr.DataArray:
        return da.sel(lon=slice(160, 210), lat=slice(-5, 5)).mean(dim=['lon', 'lat'])

    @staticmethod
    def get_nino34(da: xr.DataArray) -> xr.DataArray:
        return da.sel(lon=slice(190, 240), lat=slice(-5, 5)).mean(dim=['lon', 'lat'])

    @staticmethod
    def xr_pcc(pred: xr.DataArray, obs: xr.DataArray, dim) -> xr.DataArray:
        num = (pred * obs).sum(dim)
        denom = np.sqrt((pred**2).sum(dim)) * np.sqrt((obs**2).sum(dim))
        return num / denom

    @staticmethod
    def xr_rmse(pred: xr.DataArray, obs: xr.DataArray, dim) -> xr.DataArray:
        return np.sqrt(((pred - obs)**2).mean(dim))

    @staticmethod
    def xr_spread_skill_mve(mu: xr.DataArray, sigma: xr.DataArray, obs: xr.DataArray, dim) -> xr.DataArray:
        var = (sigma**2).mean(dim)
        mse = ((obs - mu)**2).mean(dim)
        return np.sqrt(var / mse)

    @staticmethod
    def xr_rmse_ss(pred: xr.DataArray, obs: xr.DataArray, dim) -> xr.DataArray:
        return 1 - np.sqrt(((pred - obs)**2).mean(dim) / (obs**2).mean(dim))

    @staticmethod
    def xr_activ(pred: xr.DataArray, obs: xr.DataArray, dim) -> xr.DataArray:
        return np.sqrt((pred**2).mean(dim) / (obs**2).mean(dim))

    @staticmethod
    def xr_gaussian_crps(mu: xr.DataArray, sigma: xr.DataArray,
                         obs: xr.DataArray) -> xr.DataArray:
        sqrtPi, sqrtTwo = math.sqrt(math.pi), math.sqrt(2)
        sigma = sigma.clip(min=1e-6)
        z = (obs - mu) / sigma
        phi = np.exp(-z**2 / 2) / (sqrtTwo * sqrtPi)
        return sigma * (z * xr.apply_ufunc(erf, z / sqrtTwo) + 2 * phi - 1 / sqrtPi)

    @staticmethod
    def xr_gaussian_ign(mu: xr.DataArray, sigma: xr.DataArray,
                        obs: xr.DataArray) -> xr.DataArray:
        sigma = sigma.clip(min=1e-6)
        z = (obs - mu) / sigma
        return 0.5 * math.log(2 * math.pi) + np.log(sigma) + 0.5 * z**2

    @staticmethod
    def xr_crps_ss(mu: xr.DataArray, sigma: xr.DataArray,
                   obs: xr.DataArray, dim) -> xr.DataArray:
        crps = Evaluation.xr_gaussian_crps(mu, sigma, obs).mean(dim)
        clim = Evaluation.xr_gaussian_crps(xr.zeros_like(obs), xr.ones_like(obs), obs).mean(dim)
        return 1 - crps / clim

In [ ]:
def load_config(run_id: str, eval_data: str = None, batch_size: int = None) -> MTMConfig:
    cfg = MTMConfig.from_omegaconf(OmegaConf.load(Path('runs') / str(run_id) / 'config.yaml'))
    if eval_data:
        cfg.data.eval_data = eval_data
    if batch_size:
        cfg.world.batch_size = batch_size
    return cfg